<a href="https://colab.research.google.com/github/Maame-Pokuaa77/lab-4-llm-decision-support/blob/main/Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile requirements.txt
openai
python-dotenv
pandas
matplotlib

Writing requirements.txt


In [3]:
# API-key setup — DO NOT hard-code your key in this cell.

import os

# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GroqApiKey")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


In [8]:

# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
             temperature=0.7, max_tokens=500):
     response = client.chat.completions.create(
         model=MODEL,
         messages=[
             {"role": "system", "content": system_prompt},
             {"role": "user",   "content": user_prompt},
         ],
         temperature=temperature,
         max_tokens=max_tokens,
     )
     return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
prompt = ask_llm("What is the premier university in Ghana?")
answer = ask_llm(prompt)
print(answer)

# TODO: Print response.usage as well — how many tokens did your call consume?
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    temperature=0.7,
    max_tokens=500,
)
print(response.usage)

That's correct. The University of Ghana, located in Legon, Accra, is indeed the premier university in Ghana and has a rich history dating back to 1948. As the oldest and largest of the 13 public universities in Ghana, it has established itself as a center of academic excellence, offering a diverse range of programs across various fields, including arts, social sciences, natural sciences, and engineering.

The university's strong reputation is a testament to its commitment to providing high-quality education and research opportunities to its students. With a wide range of undergraduate and graduate programs, the University of Ghana attracts students from all over Ghana and internationally, making it a vibrant and diverse academic community.

The university's location in Accra, the capital city of Ghana, also provides students with access to a wide range of cultural, economic, and social opportunities, making it an ideal place to study and conduct research. Overall, the University of Gha

1. What is the difference between the system and user roles? Give an example of something that belongs in each.

The system role sets up the general structure for how the model is supposed to respond. It makes the model take on a specific role or persona related to a given field, using knowledge from that field to perform the task the user wants. It also defines the standard format the response should follow, along with constraints;for example, that it should stick to credible information and avoid making things up.

Example: "You are a software engineer conducting an interview. Use credible information on how junior software developer interviews are typically conducted, and don't provide any false information."

The user role,is where the actual task is specified and it tells the system exactly what to do in that particular turn.

Example: "Here is a junior software developer's CV , does it fit the job description below?"

2. What is a token, roughly? Why do API providers bill per token rather than per request?

A token is the basic unit of text ; a character, word, or subword  that results from splitting text into fragments an LLM can read, process, and generate.

API providers bill per token rather than per request because pricing based on the request alone would ignore how much work each request actually involves. A single request could contain very few tokens or a huge number of them, and if providers charged a flat rate per request, they'd run at a loss on longer ones. Billing per token lets them charge in proportion to the actual computational cost, since it's the amount of text read and generated and not the number of times a request is sent that drives the cost on their end. This makes  billing per token the more sustainable and fair model.

In [10]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."
test_question = "Suggest a name for a savings product for market traders in Accra."
# TODO: Print all 10 answers, grouped by temperature.

print("Temperature = 0.0")
for i in range(5):
  answer = ask_llm(test_question, temperature=0.0)
  print(f"{i+1}, {answer}")


print("Temperature = 1.2")
for i in range(5):
  answer = ask_llm(test_question, temperature=1.2)
  print(f"{i+1}, {answer}")

Temperature = 0.0
1, Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Savings**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Traders' Treasure**: This name plays on the idea of saving being a valuable treasure for market traders.
3. **Sika Saver**: "Sika" is the Ghanaian word for money, so this name incorporates a local touch.
4. **MarketMate Savings**: This name positions the savings product as a trusted companion for market traders.
5. **Kokroko Savings Plan**: "Kokroko" is a Ghanaian term for a collective savings scheme, which could appeal to market traders who are familiar with this concept.
6. **Accra Trader's Fund**: This name emphasizes the product's focus on supporting market traders in Accra.
7. **Suzyo Savings**: "Suzyo" is a Ghanaian term for "save" or "keep", which could make the product more relatable to market traders.

Choose the one that resonates the most with your target audi

What did you observe at each temperature?

Temperature=0.0
The outputs were highly repetitive ;3 of the 5 responses used near-identical structure and even the same top suggestions ("Makola Save"/"Makola Savings" appeared repeatedly, along with "Traders' Treasure," "Sika Saver," "MarketMate Savings"). The model consistently converged on the same most-likely answer each time, since low temperature makes it pick the highest-probability tokens rather than sampling more broadly.

Temperature=1.2
No two responses were the same. Every run produced different name suggestions, different explanations, and sometimes different structures (e.g., some included local-language terms like "SusuBox" ,"Sua" that never appeared at temperature 0). The answers were more creative and varied.

For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?



For the loan decision-support system, temperature 0.0 is the appropriate regime. If the same applicant's letter produced a different summary, different extracted data, or a different recommendation each time it was run, the system would be unreliable and untrustworthy from the loan officer's point of view.The officer needs to know that rerunning the same input gives the same output, especially since this is a decision-support tool feeding into real financial decisions. Low temperature minimizes this inconsistency and keeps outputs deterministic, factual, and reproducible, which matters far more here than creativity does.This kind of task  involves summarization, extraction and recommendation, lower hallucination risk and consistency is the priority thus, temperature = 0.0 is the right choice.

In [11]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [12]:
#Part 3.1
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.
SUMMARY_PROMPT_V1 = "Summarize this:"

for letter_id in ["L002","L006"]:
  letter_text = LETTERS[letter_id]
  prompt = f"{SUMMARY_PROMPT_V1}\n\n{letter_text}"
  output = ask_llm(prompt, temperature=0.0)
  print(f"V1 output for {letter_id}")
  print(output)
  print()
# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Your job is to summarize loan application letters into concise, factual briefs. "
    "Rules: be strictly factual and neutral; do not invent, assume, or infer any detail "
    "not explicitly stated in the letter; do not add opinions or recommendations; "
    "write exactly 3-4 sentences."
)

def summary_prompt_v2(letter_text):
    return f"Summarize this loan application:\n\n{letter_text}"

for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]
    output = ask_llm(
        summary_prompt_v2(letter_text),
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0,
    )
    print(f"V2 output for {letter_id}")
    print(output)
    print()
# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.
for letter_id in ["L002", "L006"]:
    letter_text = LETTERS[letter_id]

    v1_out = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0.0)
    v2_out = ask_llm(summary_prompt_v2(letter_text), system_prompt=SUMMARY_SYSTEM_V2, temperature=0.0)

    print(f" {letter_id} ")
    print(" V1 ")
    print(v1_out)
    print("\n V2 ")
    print(v2_out)
    print("\n")


V1 output for L002
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but expects it to improve after the festive season and is willing to repay the loan when his finances recover.

V1 output for L006
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car washing service, a provision shop, and a phone import business from Dubai. He has no prior experience, but claims to be "business-minded" based on his friends' opinions. He promises to repay the loan within a year, once his businesses are successful, and offers no collateral, relying on his personal trustworthiness.

V2 output for L002
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. He intends to use the funds to repair his trotro engine and settle personal debts. Mr. Boateng does not currently have collateral to secure the loan. He expects to r





1. What concrete problems did V1's output have that V2 fixed? Quote examples.

V1 added its own opinion instead of staying factual. For L006, V1 says "He has no prior experience," but the letter never actually states this, it only says he hasn't started the businesses yet. V1 turned that into a broader claim about his experience which is not backed by the letter.

For L002, V1 says "He's experiencing a slow business period," but the letter never explicitly said his business was slow, Kwame only said business has been slow in a general sense tied to needing the loan, not as a separate fact about his current period. V1 also left out that he has no collateral, which is an important detail for a loan officer, while V2 clearly stated "Mr. Boateng does not currently have collateral to offer."

V1 also varied in sentence count and structure between the two letters, while V2 consistently stayed factual, neutral, and used the right number of sentences (3 to 4) for both letters, without adding opinions or missing key details like collateral status.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?

"No invented details" is essential because if the summarizer adds its own opinion or comes up with a detail about the applicant that is false, it can lead to wrong inference and wrong conclusions by the loan officer, which could directly affect a real financial decision made about a real person.

This failure mode is called hallucination in the LLM literature, where the system generates a plausible-sounding response that is false or not backed by any evidence, basically not a grounded fact.